# EX — LLM API & Prompting Real-World Exercises

**Note:** these exercises use a `MockLLMClient` so the notebook runs without API keys/cost.
Swap in a real client (see `05_LLM_APIs_and_Prompting/API_Basics` and `Gemini_Prompting_Lab`
in this course) by keeping the same `.complete(prompt, system=None, temperature=0.7)` interface.


In [ ]:
import json, random

class MockLLMClient:
    """Deterministic stand-in for a real LLM API so exercises run offline."""
    def complete(self, prompt, system=None, temperature=0.7):
        p = prompt.lower()
        if "json" in p and "sentiment" in p:
            sentiment = random.choice(["positive","negative","neutral"])
            return json.dumps({"sentiment": sentiment, "confidence": round(random.uniform(0.6,0.99), 2)})
        if "classify" in p and "ticket" in p:
            return random.choice(["billing", "technical", "account", "other"])
        return f"[mock completion for prompt of length {len(prompt)}]"

client = MockLLMClient()
print(client.complete("Classify this support ticket: 'I can't log in'"))


## 1. System vs. User Prompts & Output Format
**Pointer:** be explicit about output format — don't hope the model guesses right.

In [ ]:
system_prompt = "You are a support ticket classifier. Respond with exactly one word: billing, technical, account, or other."
ticket = "My card was charged twice for the same order."
prompt = f"{system_prompt}\n\nTicket: {ticket}"
print(client.complete(prompt))


### TODO 1
Write a prompt asking for **structured JSON** sentiment analysis of a product review, with fields `sentiment` and `confidence`. Parse the result with `json.loads`.

In [ ]:
review = "This product broke after two days, very disappointed."
# TODO: build `prompt` that clearly instructs JSON output with sentiment+confidence, call client.complete, parse json
prompt = None
result = None
parsed = None
print(parsed)


<details><summary>Solution</summary>

```python
prompt = (
    "Analyze the sentiment of this product review. "
    "Respond with ONLY valid JSON: {\"sentiment\": \"positive|negative|neutral\", \"confidence\": 0.0-1.0}.\n\n"
    f"Review: {review}"
)
result = client.complete(prompt)
parsed = json.loads(result)
```
</details>


## 2. Few-Shot Prompting
**Pointer:** examples in the prompt are often more reliable than long instructions for consistent formatting.

In [ ]:
few_shot_prompt = '''Classify the support ticket into: billing, technical, account, other.

Ticket: "I was charged the wrong amount"
Label: billing

Ticket: "The app crashes on startup"
Label: technical

Ticket: "I can't remember my password"
Label: account

Ticket: "Do you have a mobile app?"
Label: other

Ticket: "My subscription renewed but I cancelled it last month"
Label:'''
print(client.complete(few_shot_prompt))


## 3. Context Window Management — Summarizing a Long Conversation
**Pointer:** in production you can't resend an unbounded conversation; summarize old turns.

In [ ]:
conversation = [
    {"role": "user", "content": "I want to return my order"},
    {"role": "assistant", "content": "Sure, what's the order number?"},
    {"role": "user", "content": "12345"},
    {"role": "assistant", "content": "Found it. Reason for return?"},
    {"role": "user", "content": "Wrong size"},
]

def render_conversation(turns):
    return "\n".join(f"{t['role']}: {t['content']}" for t in turns)

print(render_conversation(conversation))


### TODO 2
Write a function `summarize_if_long(turns, max_turns=4)` that, if `len(turns) > max_turns`, replaces all but the last 2 turns with a single summarizer-LLM turn (use `client.complete` with a 'summarize this conversation' prompt), keeping recent turns verbatim.

In [ ]:
# TODO
def summarize_if_long(turns, max_turns=4):
    pass

new_turns = summarize_if_long(conversation, max_turns=4)
print(new_turns)


<details><summary>Solution</summary>

```python
def summarize_if_long(turns, max_turns=4):
    if len(turns) <= max_turns:
        return turns
    old, recent = turns[:-2], turns[-2:]
    summary_prompt = "Summarize this conversation in one sentence:\n" + render_conversation(old)
    summary = client.complete(summary_prompt)
    return [{"role": "system", "content": f"Earlier conversation summary: {summary}"}] + recent
```
</details>


## 4. Basic Retry/Error Handling
**Pointer:** real APIs fail (rate limits, timeouts) — always handle it.

In [ ]:
import time

class FlakyLLMClient(MockLLMClient):
    def __init__(self, fail_times=2):
        self.calls = 0
        self.fail_times = fail_times
    def complete(self, prompt, system=None, temperature=0.7):
        self.calls += 1
        if self.calls <= self.fail_times:
            raise TimeoutError("simulated timeout")
        return super().complete(prompt, system, temperature)

flaky = FlakyLLMClient(fail_times=2)


### TODO 3
Write `call_with_retry(client, prompt, max_retries=3)` that retries on `TimeoutError` with a short sleep, and raises after `max_retries` failed attempts.

In [ ]:
# TODO
def call_with_retry(client, prompt, max_retries=3):
    pass

print(call_with_retry(flaky, "hello"))


<details><summary>Solution</summary>

```python
def call_with_retry(client, prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.complete(prompt)
        except TimeoutError:
            if attempt == max_retries - 1:
                raise
            time.sleep(0.1)
```
</details>


## Key Takeaways
- Explicit output-format instructions + few-shot examples = reliable structured output.
- You must manage conversation/context length yourself — nothing is remembered for free.
- Always wrap API calls in retry logic for transient failures.
- Validate/parse model output defensively (try/except around `json.loads`).
